In [117]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

 
import seaborn as sns
from matplotlib import pyplot as plt
%matplotlib inline

from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import mean_squared_error
from sklearn.tree import export_text



In [118]:

data = "https://raw.githubusercontent.com/alexeygrigorev/datasets/master/car_fuel_efficiency.csv"
!wget $data -O car_fuel_efficiency.csv

df = pd.read_csv('car_fuel_efficiency.csv')
df.head()

--2025-11-03 21:26:50--  https://raw.githubusercontent.com/alexeygrigorev/datasets/master/car_fuel_efficiency.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 874188 (854K) [text/plain]
Saving to: ‘car_fuel_efficiency.csv’

car_fuel_efficiency 100%[===================>] 853.70K  --.-KB/s    in 0.04s   

2025-11-03 21:26:50 (22.2 MB/s) - ‘car_fuel_efficiency.csv’ saved [874188/874188]



,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,model_year,origin,fuel_type,drivetrain,num_doors,fuel_efficiency_mpg
0,170,3.0,159.0,3413.433759,17.7,2003,Europe,Gasoline,All-wheel drive,0.0,13.231729
1,130,5.0,97.0,3149.664934,17.8,2007,USA,Gasoline,Front-wheel drive,0.0,13.688217
2,170,NaN,78.0,3079.038997,15.1,2018,Europe,Gasoline,Front-wheel drive,0.0,14.246341
3,220,4.0,NaN,2542.392402,20.2,2009,USA,Diesel,All-wheel drive,2.0,16.912736
4,210,1.0,140.0,3460.870990,14.4,2009,Europe,Gasoline,All-wheel drive,2.0,12.488369


In [119]:
# fill missing values with 0
df.fillna(0, inplace=True)


In [120]:
df.head()

,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,model_year,origin,fuel_type,drivetrain,num_doors,fuel_efficiency_mpg
0,170,3.0,159.0,3413.433759,17.7,2003,Europe,Gasoline,All-wheel drive,0.0,13.231729
1,130,5.0,97.0,3149.664934,17.8,2007,USA,Gasoline,Front-wheel drive,0.0,13.688217
2,170,0.0,78.0,3079.038997,15.1,2018,Europe,Gasoline,Front-wheel drive,0.0,14.246341
3,220,4.0,0.0,2542.392402,20.2,2009,USA,Diesel,All-wheel drive,2.0,16.912736
4,210,1.0,140.0,3460.870990,14.4,2009,Europe,Gasoline,All-wheel drive,2.0,12.488369


In [121]:
# setup train, val, test splits
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)
 
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)
# Define your target column
target = 'fuel_efficiency_mpg'

# Separate features and target for each split
X_train = df_train.drop(columns=[target])
y_train = df_train[target]


X_val = df_val.drop(columns=[target])
y_val = df_val[target]

X_test = df_test.drop(columns=[target])
y_test = df_test[target]


In [122]:
# Identify numeric and categorical columns
numeric_features = ["engine_displacement", "num_cylinders", "horsepower",
                    "vehicle_weight", "acceleration", "model_year", "num_doors"]

categorical_features = ["origin", "fuel_type", "drivetrain"]

In [123]:
# remove categorial columns 
df_val = df_val.drop(columns=categorical_features)
df_train = df_train.drop(columns=categorical_features)
df_test = df_test.drop(columns=categorical_features)

In [124]:
df_train.head()

,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,model_year,num_doors,fuel_efficiency_mpg
0,120,5.0,169.0,2966.679505,13.9,2005,-1.0,15.301475
1,200,3.0,143.0,2950.822121,17.1,2013,-1.0,15.331215
2,180,6.0,180.0,3078.221669,17.4,2007,0.0,15.336679
3,280,5.0,174.0,2797.991793,0.0,2016,0.0,15.865850
4,250,4.0,133.0,2362.426930,16.3,2010,-1.0,18.102203


In [125]:
# one-hot encode categorical variables
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ],
    remainder='passthrough'  # Keep numeric columns as they are
)


In [126]:
# Question 1 Let's train a decision tree regressor to predict the fuel_efficiency_mpg variable.
# Which feature is used for splitting the data?

In [127]:

train_dicts = df_train.fillna(0).to_dict(orient='records')
train_dicts[:5]
dv = DictVectorizer(sparse=True)
dt = DecisionTreeRegressor(random_state=1, max_depth=1)

# keep dv fitted 
X_train = dv.fit_transform(train_dicts)
dt.fit(X_train, y_train)

model = DecisionTreeRegressor(random_state=1, max_depth=1)

model.fit(X_train, y_train)




,criterion,'squared_error'
,splitter,'best'
,max_depth,1
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,1
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,ccp_alpha,0.0


In [128]:
# print(export_text(dt))
names = dv.get_feature_names_out().tolist()
print(export_text(dt, feature_names=names))

|--- fuel_efficiency_mpg <= 14.96
|   |--- value: [12.94]
|--- fuel_efficiency_mpg >  14.96
|   |--- value: [16.98]



In [129]:
# Question 1 Let's train a decision tree regressor to predict the fuel_efficiency_mpg variable.
# Which feature is used for splitting the data?
# vehicle_weight

In [130]:
# Question 2
# Train a random forest regressor with these parameters:

# n_estimators=10
# random_state=1
# n_jobs=-1 (optional - to make training faster)
# What's the RMSE of this model on the validation data?
# Answer = 9.36 4.5


In [131]:
model = RandomForestRegressor(n_estimators=10, random_state=1,n_jobs=-1)
feature_names = dv.get_feature_names_out()

model.fit(X_train, y_train)

# Make predictions on the validation set
y_pred = model.predict(df_val)


# Compute RMSE
mse = mean_squared_error(y_val, y_pred)
rmse = np.sqrt(mse)
print(f"RMSE on validation data: {rmse:.2f}")


RMSE on validation data: 9.36


/home/codespace/.local/lib/python3.12/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(


In [132]:
# Question 3
# Now let's experiment with the n_estimators parameter

# Try different values of this parameter from 10 to 200 with step 10.
# Set random_state to 1.
# Evaluate the model on the validation dataset.
# Answer = 80

In [ ]:
estimators = list(range(10, 201, 10))
scores = []
 
for e in estimators:
    model = RandomForestRegressor(n_estimators=e, random_state=1,n_jobs=-1)

    model.fit(X_train, y_train)

    # Make predictions on the validation set
    y_pred = model.predict(df_val)
    # Compute RMSE
    mse = mean_squared_error(y_val, y_pred)
    rmse = np.sqrt(mse)
    scores.append((e, rmse.round(3)))
columns = ['n_estimators', 'rsme']
df_scores = pd.DataFrame(scores, columns=columns)
df_scores
    

In [134]:
#  Question 4
# Let's select the best max_depth:

# Try different values of max_depth: [10, 15, 20, 25]
# For each of these values,
# try different values of n_estimators from 10 till 200 (with step 10)
# calculate the mean RMSE
# Fix the random seed: random_state=1
# What's the best max_depth, using the mean RMSE?
# Answer 10 4.22

In [ ]:
estimators = list(range(10, 201, 10))
depths = [10, 15, 20, 25]
scores = []
 
for d in depths:
  for e in estimators:
      model = RandomForestRegressor(n_estimators=e, random_state=1,n_jobs=-1)
      model.fit(X_train, y_train)
      # Make predictions on the validation set
      y_pred = model.predict(df_val)
      # Compute RMSE
      mse = mean_squared_error(y_val, y_pred)
      rmse = np.sqrt(mse)
      scores.append((e, d, rmse.round(3)))
columns = ['estimators', 'depth', 'rsme']
df_scores = pd.DataFrame(scores, columns=columns)
df_scores

In [136]:
# Question 5
# Train the model with these parameters:
# n_estimators=10,
# max_depth=20,
# random_state=1,
# n_jobs=-1 (optional)
# Get the feature importance information from this model
# What's the most important feature (among these 4)?

# vehicle_weight
# horsepower
# acceleration
# engine_displacement
# answer acceleration

In [137]:
model = RandomForestRegressor(n_estimators=10, max_depth=20, random_state=1,n_jobs=-1)

model.fit(X_train, y_train)
# Get feature importances
importances = model.feature_importances_
print(importances)
feature_names = ['vehicle_weight',
                'horsepower',
                'acceleration',
                'engine_displacement']
print(feature_names)

# Print feature importances matched to names
for name, importance in zip(feature_names, importances):
    print(f"{name}: {importance:.4f}")

[2.55267172e-05 1.64970155e-05 9.99760281e-01 1.59264435e-05
 3.53902604e-06 1.14863410e-06 2.56376210e-05 1.51443294e-04]
['vehicle_weight', 'horsepower', 'acceleration', 'engine_displacement']
vehicle_weight: 0.0000
horsepower: 0.0000
acceleration: 0.9998
engine_displacement: 0.0000


In [ ]:
# Question 6
# Now let's train an XGBoost model! For this question, we'll tune the eta parameter:

# Install XGBoost
# Create DMatrix for train and validation
# Create a watchlist
# Train a model with these parameters for 100 rounds:
# 'eta': 0.3 rmse 0.452
# 'eta': 0.1 rmse 0.429 this is the answer

In [139]:
%pip install xgboost
import sys
!{sys.executable} -m pip install xgboost
import xgboost as xgb

# https://stackoverflow.com/questions/44856105/cannot-import-xgboost-in-jupyter-notebook


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip


In [147]:
xgb_params = {
    'eta': 0.3, 
    'max_depth': 6,
    'min_child_weight': 1,
     
    'objective': 'reg:squarederror',
    'nthread': 8,
     
    'seed': 1,
    'verbosity': 1,
}
	

dtrain = xgb.DMatrix(df_train.drop(columns=[target]).values, label=df_train[target].values)
dval   = xgb.DMatrix(df_val.drop(columns=[target]).values,   label=df_val[target].values)
	
watchlist = [(dtrain, 'train'), (dval, 'val')]
model = xgb.train(xgb_params, dtrain, num_boost_round=100,
                  evals=watchlist)
y_pred = model.predict(dval)
# Compute RMSE
mse = mean_squared_error(y_val, y_pred)
rmse = np.sqrt(mse)
print(rmse.round(3))

[0]	train-rmse:1.81393	val-rmse:1.85444
[1]	train-rmse:1.31919	val-rmse:1.35353
[2]	train-rmse:0.98120	val-rmse:1.01316
[3]	train-rmse:0.75443	val-rmse:0.78667
[4]	train-rmse:0.60680	val-rmse:0.64318
[5]	train-rmse:0.51389	val-rmse:0.55708
[6]	train-rmse:0.45505	val-rmse:0.50368
[7]	train-rmse:0.42024	val-rmse:0.47227
[8]	train-rmse:0.39713	val-rmse:0.45463
[9]	train-rmse:0.38295	val-rmse:0.44441
[10]	train-rmse:0.37374	val-rmse:0.43904
[11]	train-rmse:0.36612	val-rmse:0.43585
[12]	train-rmse:0.36141	val-rmse:0.43450
[13]	train-rmse:0.35683	val-rmse:0.43247
[14]	train-rmse:0.35358	val-rmse:0.43218
[15]	train-rmse:0.35082	val-rmse:0.43191
[16]	train-rmse:0.34776	val-rmse:0.43323
[17]	train-rmse:0.34483	val-rmse:0.43413
[18]	train-rmse:0.34192	val-rmse:0.43432
[19]	train-rmse:0.33869	val-rmse:0.43477
[20]	train-rmse:0.33698	val-rmse:0.43516
[21]	train-rmse:0.33650	val-rmse:0.43534
[22]	train-rmse:0.33522	val-rmse:0.43585
[23]	train-rmse:0.33168	val-rmse:0.43538
[24]	train-rmse:0.32935	va